In [1]:
import uuid
import os
import tensorflow as tf
import pandas as pd
from code import *
from tensorflow.keras.models import Model
import numpy as np
import re

In [2]:
os.chdir("../../")

# Situation 1: input the peptide of length 39 and the sequences only contain 20 standard amino acids

## Step 1: load dataset

In [44]:
seqs = ['LEKELFREMESILQNKHLDVEKIVNLFPQCT', 'SSTASSMELEELRHEKEMQREEIQKLMGQIH', 'TPKKAKKAAGAKKAVKKTPKKAKKPAAAGVK']

## Step 2: extract features

In [45]:
protein_bert_test = extract_embedding_features(seqs)
tf.keras.backend.clear_session()
BLOSUM62_test = BLOSUM62(seqs)
one_hot_test = np.array(to_embedding_numeric(seqs)).astype(np.float32)

## Step 3: load models

In [46]:
model_1 = CNN(protein_bert_test)
model_2 = BiGRU()
model_3 = CNN(BLOSUM62_test)
model_atnn = ensemble_model()

In [47]:
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')

## Step 4: predict and save results

In [48]:
ensemble_valid_input = np.concatenate((model_1.predict(protein_bert_test, verbose=0),
                                  model_2.predict(one_hot_test, verbose=0),
                                  model_3.predict(BLOSUM62_test, verbose=0)), axis=-1)

In [49]:
df = pd.DataFrame(seqs, columns=['Sequnce'])

In [50]:
p = model_atnn.predict(ensemble_valid_input, verbose=0)
prediction = np.zeros_like(p)
prediction[p >= 0.5] = 1
df['probability'] = p
df['Label'] = prediction.astype(int)

In [51]:
df

,Sequnce,probability,Label
0,LEKELFREMESILQNKHLDVEKIVNLFPQCT,0.806675,1
1,SSTASSMELEELRHEKEMQREEIQKLMGQIH,0.617796,1
2,TPKKAKKAAGAKKAVKKTPKKAKKPAAAGVK,0.021739,0


# Situation 2: input the protein sequence

In [52]:
def construct_dataset(seqs):
    uid = uuid.uuid4()
    file_name = str(uid) + '.csv'
    file_path = os.path.join('code/demo/prediction/', file_name)
    f = open(file_path, 'w')
    f.write('protein_id,Sequence, Location\n')
    for i, seq in enumerate(seqs):
        l = len(seq)
        frags = []
        locs = []
        for pos in range(l):
            if 'K' == seq[pos]:
                prefix = 'X' * (15 - pos) + seq[0: pos] if pos < 15 else seq[pos - 15: pos]
                suffix = seq[pos + 1:] + 'X' * (15 - (l - pos - 1)) if (l - pos - 1) < 15 else seq[pos + 1: pos + 16]
                frag = prefix + 'K' + suffix
                frags.append(frag)
                locs.append(pos)
        for pos, frag in zip(locs, frags):
            line = f'{i},{frag},{pos}\n'
            f.write(line)
    return file_path

In [53]:
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

In [54]:
def extract_word_embedding_features(seqs, embedding_layer):
    voc = np.array(to_embedding_numeric(seqs)).astype(np.int32)
    weights = embedding_layer.weights[0].numpy()
    x = np.mean(weights, axis=0, keepdims=True) # adopt the mean as the vector for the padding 'X'
    weights = np.concatenate([weights, x], axis=0)
    return weights[voc]

## Step 1: load protein sequences

In [55]:
seqs = ['MAQKENSYPWPYGRQTAPSGLSTLPQRVLRKEPVTPSALVLMSRSNVQPTAAPGQKVMENSSGTPDILTRHFTIDDFEIGRPLGKGKFGNVYLAREKKSHFIVALKVLFKSQIEKEGVEHQLRREIEIQAHLHHPNILRLYNYFYDRRRIYLILEYAPRGELYKELQKSCTFDEQRTATIMEELADALMYCHGKKVIHRDIKPENLLLGLKGELKIADFGWSVHAPSLRRKTMCGTLDYLPPEMIEGRMHNEKVDLWCIGVLCYELLVGNPPFESASHNETYRRIVKVDLKFPASVPMGAQDLISKLLRHNPSERLPLAQVSAHPWVRANSRRVLPPSALQSVA']

## Step 2: split protein sequences

In [56]:
file_path = construct_dataset(seqs)

## Step 3: predict

In [57]:
df = pd.read_csv(file_path, sep=',', header=0)
seqs = df['Sequence']

In [58]:
pbf = extract_embedding_features(seqs)
blosum62_f = BLOSUM62(seqs)

In [59]:
model_1 = CNN(pbf)
model_2 = BiGRU()
model_3 = CNN(blosum62_f)
model_atnn = ensemble_model()
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')
embedding_layer = model_2.layers[1]
model_2 = Model(inputs=[model_2.layers[2].input], outputs=[model_2.output])

In [60]:
wef = extract_word_embedding_features(seqs, embedding_layer)

In [61]:
p_1 = model_1.predict(pbf, verbose=0)
p_2 = model_2.predict(wef, verbose=0)
p_3 = model_3.predict(blosum62_f, verbose=0)
p = model_atnn.predict(np.concatenate([p_1, p_2, p_3], axis=1), verbose=0)
prediction = np.zeros_like(p)
prediction[p >= 0.5] = 1
df['probability'] = p
df['Label'] = prediction.astype(int)

In [62]:
df

,protein_id,Sequence,Location,probability,Label
0,0,XXXXXXXXXXXXMAQKENSYPWPYGRQTAPS,3,0.028564,0
1,0,TAPSGLSTLPQRVLRKEPVTPSALVLMSRSN,30,0.057246,0
2,0,LMSRSNVQPTAAPGQKVMENSSGTPDILTRH,55,0.207995,0
3,0,RHFTIDDFEIGRPLGKGKFGNVYLAREKKSH,84,0.593506,1
4,0,FTIDDFEIGRPLGKGKFGNVYLAREKKSHFI,86,0.722943,1
5,0,PLGKGKFGNVYLAREKKSHFIVALKVLFKSQ,96,0.428015,0
6,0,LGKGKFGNVYLAREKKSHFIVALKVLFKSQI,97,0.527416,1
7,0,VYLAREKKSHFIVALKVLFKSQIEKEGVEHQ,105,0.304761,0
8,0,REKKSHFIVALKVLFKSQIEKEGVEHQLRRE,109,0.800468,1
9,0,HFIVALKVLFKSQIEKEGVEHQLRREIEIQA,114,0.932975,1
